In [10]:
import urllib
import time
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd

DRIVER_PATH = 'chromedriver.exe'
BASE_URL = 'https://map.naver.com/v5/search/'

def collectFamousStore(station):
    
    # 키워드 생성 후 URL로 변환
    keyword = station + "맛집"
    
    encText = urllib.parse.quote(keyword)
    url = BASE_URL + encText
    
    # selenium 브라우저로 해당 URL 접속
    browser = webdriver.Chrome(DRIVER_PATH)
    time.sleep(2)
    browser.get(url)

    # 필요한 정보가 있는 iframe 페이지에 name 속성명을 통한 접근
    browser.switch_to.frame('searchIframe')
    time.sleep(2)

    # 데이터를 담을 리스트와 각 항목에 대한 접근을 확인하기 위한 cnt 변수 생성
    outList = []
    cnt = 0
    
    while True:
        
        # 스크롤을 하면 추가되는 <li> 항목을 모두 가져오기 위한 div 스크롤 조작  
        container = browser.find_element_by_id('_pcmap_list_scroll_container')
        browser.execute_script("arguments[0].scrollBy(0, 3000)", container)                                
        time.sleep(2)
        browser.execute_script("arguments[0].scrollBy(0, 4500)", container)
        time.sleep(2)
        browser.execute_script("arguments[0].scrollBy(0, 5000)", container)
        time.sleep(2)
        
        # 스크롤 된 페이지 소스를 bs4 객체로 변환
        htmlStr = browser.page_source
        bs = BeautifulSoup(htmlStr,'html.parser')
        
        targetDiv = bs.select_one('div#_pcmap_list_scroll_container')
        targets = targetDiv.select("li[class='_1EKsQ _12tNp']")
        
        # 페이지당 50개인 <li> 항목을 모두 가져왔는지 확인 
        print(len(targets))

        for t in targets:
            
            # 접근 중인 항목 번호 
            cnt += 1
            print(cnt)
            
            # 점포명
            name = t.select_one('span.place_bluelink').text

            # 별점 (별점이 없으면 None을 저장)
            score = t.select_one('em')
            if score:
                score = float(score.text)
            else:
                score = None
            
            # 썸네일 이미지
            img = t.select_one('div.cb7hz')
            if img:
                img = img['style'].split('"')[1]
            else:
                img = None
            
            # 영업 상태, 별점, 리뷰 등이 담겨있는 div 태그 텍스트
            info = t.select_one('div._17H46')
            
            if info:
                info = info.text
                
                # 블로그리뷰 를 기준으로 split한 리스트의 길이가 1이면
                # 블로그리뷰에 대한 정보가 없는 것이므로 None을 저장
                if len(info.split('블로그리뷰'))== 1:
                    blogNum = None
                    
                    if len(info.split('방문자리뷰'))== 1:
                        visitNum = None

                    else:
                    
                    # 리뷰 수가 1000 이상이면 ,로 구분되므로 제거
                        info = info.split('방문자리뷰')
                        visitNum = int(info[-1].replace(',',''))
                    
                else:
                    info = info.split('블로그리뷰')
                    blogNum = int(info[-1].replace(',',''))

                    # split된 텍스트 중 나머지 정보가 담긴 텍스트로 이동
                    info = info[0]
                    
                    if len(info.split('방문자리뷰'))== 1:
                        visitNum = None

                    else:
                        info = info.split('방문자리뷰')
                        visitNum = int(info[-1].replace(',',''))
                        
            else:
                blogNum = None
                visitNum = None
            
            if score and visitNum and blogNum:
                if visitNum >= 2000:
                    totalScore1 = 1
                elif visitNum >= 1250:
                    totalScore1 = 0.9
                elif visitNum >= 750:
                    totalScore1 = 0.8
                elif visitNum >= 600:
                    totalScore1 = 0.7
                elif visitNum >= 500:
                    totalScore1 = 0.6
                elif visitNum >= 400:
                    totalScore1 = 0.5
                elif visitNum >= 300:
                    totalScore1 = 0.4
                elif visitNum >= 200:
                    totalScore1 = 0.3
                else:
                    totalScore1 = 0.1

                if blogNum >= 2000:
                    totalScore2 = 1
                elif blogNum >= 1250:
                    totalScore2 = 0.9
                elif blogNum >= 750:
                    totalScore2 = 0.8
                elif blogNum >= 600:
                    totalScore2 = 0.7
                elif blogNum >= 500:
                    totalScore2 = 0.6
                elif blogNum >= 400:
                    totalScore2 = 0.5
                elif blogNum >= 300:
                    totalScore2 = 0.4
                elif blogNum >= 200:
                    totalScore2 = 0.3
                else:
                    totalScore2 = 0.1

                totalScore = (totalScore1 * 0.7) + (totalScore2 * 0.3) + 4 + score
                
            else:
                totalScore = None
            
            
            outList.append([name, score, img, visitNum, blogNum, totalScore])

            
        # 탐색이 끝난 후 다음 페이지로 이동
        
        # 이전 페이지, 5개의 페이지 번호, 다음 페이지 버튼으로 구성된
        # 7개의 <a> 태그 중 다음 페이지 버튼인 마지막 <a> 태그 선택
        nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]
        
        # 해당 버튼의 class명 확인
        isLastPage = nextBtn.get_attribute("class")
        print(isLastPage)

        # 클래스명이 "_2bgjK "이면 (공백 포함 주의 !!)
        # 마지막 페이지가 아니므로 다음 페이지로 이동
        if isLastPage == "_2bgjK ":
            print("this is not last page.")
            nextBtn.click()
            time.sleep(2)
        
        # class명이 "_2bgjK _34lTS"이면 마지막 페이지이므로 반복문 종료
        else:
            break

    # 엑셀에서 한글이 깨지지 않는 'utf-8-sig' 인코딩으로 csv 파일 추출 
    df = pd.DataFrame(outList,columns=['점포명','별점','이미지','방문자 리뷰 수', '블로그 리뷰 수', '평점'])
    df.to_csv(f"{keyword}CSV.csv",encoding='utf-8-sig')
    
if __name__ == "__main__":
#     collectFamousStore(input())
    stationList = ['범내골역','범일역','좌천역','부산진역','초량역','부산역','중앙역','남포역','자갈치역']
    #지하철역명 입력 후 collectFamousStore 함수로 전달 
    for station in stationList:
        collectFamousStore(station)

C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:19: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  browser = webdriver.Chrome(DRIVER_PATH)
C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:34: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  container = browser.find_element_by_id('_pcmap_list_scroll_container')


50
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
_2bgjK 
this is not last page.


C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:165: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]


50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
_2bgjK 
this is not last page.
50
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
_2bgjK 
this is not last page.
50
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
_2bgjK 
this is not last page.
50
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
_2bgjK 
this is not last page.
50
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27

C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:19: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  browser = webdriver.Chrome(DRIVER_PATH)
C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:34: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  container = browser.find_element_by_id('_pcmap_list_scroll_container')


...

50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
_2bgjK 
this is not last page.
50
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
_2bgjK 
this is not last page.
50
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
_2bgjK 
this is not last page.
50
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
_2bgjK 
this is not last page.
50
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27

C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:19: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  browser = webdriver.Chrome(DRIVER_PATH)
C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:34: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  container = browser.find_element_by_id('_pcmap_list_scroll_container')


50
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
_2bgjK 
this is not last page.


C:\Users\pnu\AppData\Local\Temp\ipykernel_2600\411976981.py:165: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]


50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
_2bgjK 
this is not last page.
50
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
_2bgjK 
this is not last page.
50
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
_2bgjK 
this is not last page.
50
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
_2bgjK 
this is not last page.
50
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27